# 03b · 5.ª familia: modelo preentrenado (embeddings) y comparación de candidatos
Se usa un modelo de oraciones multilingüe **preentrenado** (no se ajusta con nuestros datos, así que no hay fuga) y sobre sus embeddings una regresión logística. Misma validación y métrica que los demás.
Ejecutar después de `03`. La primera vez descarga el modelo y calcula los embeddings (varios minutos en CPU).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns, mlflow
from sklearn.linear_model import LogisticRegression
from src.datos import cargar_obt
from src.pipeline import *
from src.experimento import *
sns.set_theme(style="whitegrid")
df = agregar_features(cargar_obt())
train, test = split_temporal(df, 0.2)
X, y = train[FEATURES], train[TARGET]
FECHA_CORTE = df[FECHA].max().isoformat()
configurar()

In [ ]:
limpiar_fase("candidato", familias=["estado_del_arte"])
r = correr("candidato-Embeddings+LogReg",
           construir_pipeline(LogisticRegression(max_iter=1000), texto="embeddings"),
           "candidato", "estado_del_arte", X, y,
           {"modelo": "LogisticRegression", "embeddings": EmbeddingTransformer().modelo, "max_iter": 1000}, FECHA_CORTE)
pd.Series({k: v for k, v in r.items() if k not in ('run_id', 'nombre', 'model_version')}).round(4)

## Comparación de todos los candidatos (leída desde MLflow) y elección de los 3 a ajustar

In [ ]:
exp = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
runs = mlflow.search_runs([exp.experiment_id])
runs = runs[runs["tags.fase"].isin(["baseline", "candidato"])]
t = runs.set_index("tags.mlflow.runName")[["tags.fase","tags.familia","metrics.train_pr_auc","metrics.cv_pr_auc_mean","metrics.cv_pr_auc_std","metrics.brecha_pr_auc","metrics.costo_cv_umbral","metrics.segundos_entrenamiento"]]
t.columns = ["fase","familia","train_pr_auc","cv_pr_auc_mean","cv_pr_auc_std","brecha","costo_cv","seg_entren"]
t = t.sort_values("cv_pr_auc_mean", ascending=False).round(4)
display(t)